In [1]:
def format_time(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    
    time_str = ""
    if hours > 0:
        time_str += f"{hours}h"
    if minutes > 0:
        time_str += f"{minutes}min"
    if seconds > 0:
        time_str += f"{seconds}s"
    
    return time_str

# 测试
print(format_time(1))   # 输出: 1s
print(format_time(61))  # 输出: 1min1s

1s
1min1s


In [2]:
import pytest

def test_format_time():
    assert format_time(1) == "1s"
    assert format_time(61) == "1min1s"
    assert format_time(3661) == "1h1min1s"
    assert format_time(3600) == "1h"
    assert format_time(3601) == "1h1s"
    assert format_time(3660) == "1h1min"
    assert format_time(3665) == "1h1min5s"
    assert format_time(60) == "1min"
    assert format_time(600) == "10min"
    assert format_time(0) == "0s"

In [3]:
import openai, os
from openai import OpenAI
openai.api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(base_url= "https://api.zhiyungpt.com/v1", api_key=openai.api_key)

COMPLETION_MODEL = "gpt-3.5-turbo-instruct"
    
def gpt35(prompt, model=COMPLETION_MODEL, temperature=0.4, max_tokens=1000, 
          top_p=1, stop=["\n\n", "\n\t\n", "\n    \n"]):
    response = client.completions.create(
        model=model,
        prompt=prompt,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
        stop = stop
    )
    
    return response.choices[0].text

code = """
def format_time(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    
    time_str = ""
    if hours > 0:
        time_str += f"{hours}h"
    if minutes > 0:
        time_str += f"{minutes}min"
    if seconds > 0:
        time_str += f"{seconds}s"
    
    return time_str
"""

def explain_code(function_to_test, unit_test_package="pytest"):
    prompt = f""""# How to write great unit tests with {unit_test_package}

In this advanced tutorial for experts, we'll use Python 3.10 and `{unit_test_package}` to write a suite of unit tests to verify the behavior of the following function.
```python
{function_to_test}
```

Before writing any unit tests, let's review what each element of the function is doing exactly and what the author's intentions may have been.
- First,"""
    response = gpt35(prompt)
    return response, prompt

code_explaination, prompt_to_explain_code = explain_code(code)
print(code_explaination)
print(prompt_to_explain_code)

 the function takes in a number of seconds as its only argument.
- Next, it calculates the number of hours, minutes, and seconds in that amount of time.
- Then, it creates an empty string to store the formatted time.
- Finally, it checks if there are any hours, minutes, or seconds in the time and adds them to the string if they exist.
"# How to write great unit tests with pytest

In this advanced tutorial for experts, we'll use Python 3.10 and `pytest` to write a suite of unit tests to verify the behavior of the following function.
```python

def format_time(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    
    time_str = ""
    if hours > 0:
        time_str += f"{hours}h"
    if minutes > 0:
        time_str += f"{minutes}min"
    if seconds > 0:
        time_str += f"{seconds}s"
    
    return time_str

```

Before writing any unit tests, let's review what each element of the function is doing exactly and what the author's in

In [4]:
def generate_a_test_plan(full_code_explaination, unit_test_package="pytest"):
    prompt_to_explain_a_plan = f"""
    
A good unit test suite should aim to:
- Test the function's behavior for a wide range of possible inputs
- Test edge cases that the author may not have foreseen
- Take advantage of the features of `{unit_test_package}` to make the tests easy to write and maintain
- Be easy to read and understand, with clean code and descriptive names
- Be deterministic, so that the tests always pass or fail in the same way
`{unit_test_package}` has many convenient features that make it easy to write and maintain unit tests. We'll use them to write unit tests for the function above.
For this particular function, we'll want our unit tests to handle the following diverse scenarios (and under each scenario, we include a few examples as sub-bullets):
-"""
    prompt = full_code_explaination + prompt_to_explain_a_plan
    response = gpt35(prompt)
    return response, prompt
test_plan, prompt_to_get_test_plan = generate_a_test_plan(prompt_to_explain_code + code_explaination)
print(test_plan)


 Normal cases, where the input is a positive integer
    - 3600 seconds (1 hour)
    - 3661 seconds (1 hour, 1 minute, 1 second)
    - 7200 seconds (2 hours)
- Edge cases, where the input is 0 or a negative integer
    - 0 seconds
    - -1 seconds
    - -3600 seconds (-1 hour)
- Special cases, where the input is not a valid integer
    - "abc" (a string)
    - 3.5 (a float)
    - None (a NoneType)


In [5]:
not_enough_test_plan = """The function is called with a valid number of seconds
    - `format_time(1)` should return `"1s"`
    - `format_time(59)` should return `"59s"`
    - `format_time(60)` should return `"1min"`
"""
approx_min_cases_to_cover = 7
elaboration_needed = test_plan.count("\n-") +1 < approx_min_cases_to_cover 
if elaboration_needed:
        prompt_to_elaborate_on_the_plan = f"""
In addition to the scenarios above, we'll also want to make sure we don't forget to test rare or unexpected edge cases (and under each edge case, we include a few examples as sub-bullets):
-"""
more_test_plan, prompt_to_get_test_plan = generate_a_test_plan(prompt_to_explain_code + code_explaination + not_enough_test_plan + prompt_to_elaborate_on_the_plan)
print(more_test_plan)


 The function is called with a valid number of seconds
    - `format_time(1)` should return `"1s"`
    - `format_time(59)` should return `"59s"`
    - `format_time(60)` should return `"1min"`
- The function is called with an invalid input (e.g. a string or negative number)
    - `format_time("abc")` should raise a `TypeError`
    - `format_time(-1)` should raise a `ValueError`
- The function is called with a very large number of seconds
    - `format_time(3600)` should return `"1h"`
    - `format_time(3661)` should return `"1h1min1s"`
- The function is called with a number of seconds that results in a non-integer number of hours, minutes, or seconds
    - `format_time(3665)` should return `"1h1min5s"`
    - `format_time(3601)` should return `"1h1s"`
    - `format_time(61)` should return `"1min1s"`
- The function is called with a number of seconds that results in a zero value for hours, minutes, or seconds
    - `format_time(3600)` should return `"1h"`
    - `format_time(60)` should ret

In [6]:
def generate_test_cases(function_to_test, unit_test_package="pytest"):
    starter_comment = "Below, each test case is represented by a tuple passed to the @pytest.mark.parametrize decorator"
    prompt_to_generate_the_unit_test = f"""
Before going into the individual tests, let's first look at the complete suite of unit tests as a cohesive whole. We've added helpful comments to explain what each line does.
```python
import {unit_test_package}  # used for our unit tests
{function_to_test}
#{starter_comment}"""
    full_unit_test_prompt = prompt_to_explain_code + code_explaination + test_plan + prompt_to_generate_the_unit_test
    return gpt35(model=COMPLETION_MODEL, prompt=full_unit_test_prompt, stop="```"), prompt_to_generate_the_unit_test

unit_test_response, prompt_to_generate_the_unit_test = generate_test_cases(code)
print(unit_test_response)



@pytest.mark.parametrize(
    "seconds, expected",  # the two arguments we are testing for
    [
        (3600, "1h"),  # normal case: 1 hour
        (3661, "1h1min1s"),  # normal case: 1 hour, 1 minute, 1 second
        (7200, "2h"),  # normal case: 2 hours
        (0, ""),  # edge case: 0 seconds
        (-1, ""),  # edge case: -1 seconds
        (-3600, ""),  # edge case: -1 hour
        ("abc", ""),  # special case: a string
        (3.5, ""),  # special case: a float
        (None, ""),  # special case: a NoneType
    ]
)
def test_format_time(seconds, expected):
    assert format_time(seconds) == expected  # assert that the function output matches the expected output



In [7]:
import ast
code_start_index = prompt_to_generate_the_unit_test.find("```python\n") + len("```python\n")
code_output = prompt_to_generate_the_unit_test[code_start_index:] + unit_test_response
print(code_output)
try:
    ast.parse(code_output)
except SyntaxError as e:
    print(f"Syntax error in generated code: {e}")


import pytest  # used for our unit tests

def format_time(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    seconds = seconds % 60
    
    time_str = ""
    if hours > 0:
        time_str += f"{hours}h"
    if minutes > 0:
        time_str += f"{minutes}min"
    if seconds > 0:
        time_str += f"{seconds}s"
    
    return time_str

#Below, each test case is represented by a tuple passed to the @pytest.mark.parametrize decorator
@pytest.mark.parametrize(
    "seconds, expected",  # the two arguments we are testing for
    [
        (3600, "1h"),  # normal case: 1 hour
        (3661, "1h1min1s"),  # normal case: 1 hour, 1 minute, 1 second
        (7200, "2h"),  # normal case: 2 hours
        (0, ""),  # edge case: 0 seconds
        (-1, ""),  # edge case: -1 seconds
        (-3600, ""),  # edge case: -1 hour
        ("abc", ""),  # special case: a string
        (3.5, ""),  # special case: a float
        (None, ""),  # special case: a NoneType
    ]
)


In [8]:
format_time(-1)

'59min59s'